In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
import os
import json
from copy import deepcopy

import torch
import pandas as pd
from protenix.config import parse_configs
from protenix.data.msa_featurizer import tokenize_msa
from protenix.data.tokenizer import AtomArrayTokenizer, TokenArray
from protenix.utils.torch_utils import dict_to_tensor
from protenix.utils.lmdb import LMDBDataset
from protenix.utils.torch_utils import to_device
from tqdm import tqdm

from protenix.data.screen_dataset import (
    FeatureCompressor,
    ComplexFeatureDataset,
    ComplexFeatureRandomPairDataset,
)
from protenix.model.protenis import (
    ProtenisP,
    ProtenisPCrossIndependentRanker,
)
from protenix.model.screen_loss import RankNetLoss

import sys

sys.path.append("..")
from configs.configs_base import configs as configs_base
from configs.configs_data import data_configs
from configs.configs_inference import inference_configs

In [4]:
configs_base["use_deepspeed_evo_attention"] = (
    os.environ.get("USE_DEEPSPEED_EVO_ATTTENTION", False) == "true"
)
configs_base["model"]["N_cycle"] = 10
configs_base["sample_diffusion"]["N_sample"] = 5
configs_base["sample_diffusion"]["N_step"] = 200
configs = {**configs_base, **{"data": data_configs}, **inference_configs}
configs = parse_configs(
    configs=configs,
    fill_required_with_null=True,
)

In [ ]:
torch.cuda.set_device("cuda:3")
device = torch.device("cuda:3")
model = ProtenisP(configs).cuda()
model.eval()
dataset = ComplexFeatureRandomPairDataset(
    "/data/rerank/protenix/chembl_bdb/chembl_bdb.lmdb"
)
loss_func = RankNetLoss()
for i in tqdm(range(len(dataset))):
    sample = dataset[i]
    pred_dict, log_dict = model(
        to_device(sample["input_feature_dicts"][0], device)
    )
    if i > 1:
        break

  0%|          | 0/67645 [00:00<?, ?it/s]

In [13]:
torch.cuda.set_device("cuda:3")
device = torch.device("cuda:3")
model = ProtenisPCrossIndependentRanker(configs).cuda()
model.eval()
dataset = ComplexFeatureRandomPairDataset(
    "/data/rerank/protenix/chembl_bdb/chembl_bdb.lmdb"
)
loss_func = RankNetLoss()
for i in tqdm(range(len(dataset))):
    sample = dataset[i]
    pred_dict, log_dict = model(
        to_device(sample["input_feature_dicts"], device)
    )
    loss = loss_func(
        pair1_logit=pred_dict["logits"][0],
        pair2_logit=pred_dict["logits"][1],
        pair_label=to_device(sample["pair_label"]),
    )
    if i > 1000000:
        break
    del pred_dict, log_dict, loss

  0%|          | 0/67645 [00:04<?, ?it/s]


TypeError: cannot unpack non-iterable NoneType object

In [27]:
out = model(to_device(res["input_feature_dict"], device))

In [29]:
out['s_inputs'].shape

TypeError: tuple indices must be integers or slices, not str

In [4]:
lmdb = LMDBDataset("/data/rerank/protenix/chembl_bdb/feat_parts/chembl_bdb_20250410.lmdb")
lmdb.set_default_split("chembl_bdb")
len(lmdb)

2491307

In [ ]:
dataset = ComplexFeatureDataset("/data/rerank/protenix/chembl_bdb")

In [ ]:
with open("/data/rerank/protenix/chembl_bdb/chembl_bdb_protenix.json") as f:
    data = json.load(f)

In [17]:
pd_data = pd.read_csv("/data/rerank/protenix/chembl_bdb/chembl_bdb_merged.csv")

/tmp/ipykernel_1304851/2172999105.py:1: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  pd_data = pd.read_csv("/data/rerank/protenix/chembl_bdb/chembl_bdb_merged.csv")


In [21]:
# row 90
pd_data.iloc[90]

Database_Assay_ID                                                340-1
SMILES               Cc1ccccc1CNC(=O)[C@H]1N(CSC1(C)C)C(=O)[C@@H](O...
Uniprot_ID                                                      P03367
Standard_Type                                                       Ki
Standard_Relation                                                    =
Standard_Value                                                    0.11
Standard_Units                                                      nM
Database_Source                                              BindingDB
Patent_ID                                                          NaN
Journal                                                   Biochemistry
Name: 90, dtype: object

In [19]:
pd_data.iloc[94]

Database_Assay_ID                                                340-1
SMILES               Cc1ccccc1CNC(=O)[C@H]1N(CSC1(C)C)C(=O)[C@@H](O...
Uniprot_ID                                                      P03367
Standard_Type                                                       Ki
Standard_Relation                                                    =
Standard_Value                                                    0.11
Standard_Units                                                      nM
Database_Source                                              BindingDB
Patent_ID                                                          NaN
Journal                                                   Biochemistry
Name: 94, dtype: object

In [16]:
data[94]

{'sequences': [{'proteinChain': {'sequence': 'MGARASVLSGGELDRWEKIRLRPGGKKKYKLKHIVWASRELERFAVNPGLLETSEGCRQILGQLQPSLQTGSEELRSLYNTVATLYCVHQRIEIKDTKEALDKIEEEQNKSKKKAQQAAADTGHSSQVSQNYPIVQNIQGQMVHQAISPRTLNAWVKVVEEKAFSPEVIPMFSALSEGATPQDLNTMLNTVGGHQAAMQMLKETINEEAAEWDRVHPVHAGPIAPGQMREPRGSDIAGTTSTLQEQIGWMTNNPPIPVGEIYKRWIILGLNKIVRMYSPTSILDIRQGPKEPFRDYVDRFYKTLRAEQASQEVKNWMTETLLVQNANPDCKTILKALGPAATLEEMMTACQGVGGPGHKARVLAEAMSQVTNSATIMMQRGNFRNQRKIVKCFNCGKEGHIARNCRAPRKKGCWKCGKEGHQMKDCTERQANFLREDLAFLQGKAREFSSEQTRANSPTISSEQTRANSPTRRELQVWGRDNNSLSEAGADRQGTVSFNFPQITLWQRPLVTIKIGGQLKEALLDTGADDTVLEEMSLPGRWKPKMIGGIGGFIKVRQYDQILIEICGHKAIGTVLVGPTPVNIIGRNLLTQIGCTLNFPISPIETVPVKLKPGMDGPKVKQWPLTEEKIKALVEICTEMEKEGKISKIGPENPYNTPVFAIKKKDSTKWRKLVDFRELNKRTQDFWEVQLGIPHPAGLKKKKSVTVLDVGDAYFSVPLDEDFRKYTAFTIPSINNETPGIRYQYNVLPQGWKGSPAIFQSSMTKILEPFRKQNPDIVIYQYMDDLYVGSDLEIGQHRTKIEELRQHLLRWGLTTPDKKHQKEPPFLWMGYELHPDKWTVQPIVLPEKDSWTVNDIQKLVGKLNWASQIYPGIKVRQLCKLLRGTKALTEVIPLTEEAELELAENREILKEPVHGVYYDPSKDLIAEIQKQGQGQWTYQIYQEPFKNLKTGKYA

In [15]:
data[90]

{'sequences': [{'proteinChain': {'sequence': 'MGARASVLSGGELDRWEKIRLRPGGKKKYKLKHIVWASRELERFAVNPGLLETSEGCRQILGQLQPSLQTGSEELRSLYNTVATLYCVHQRIEIKDTKEALDKIEEEQNKSKKKAQQAAADTGHSSQVSQNYPIVQNIQGQMVHQAISPRTLNAWVKVVEEKAFSPEVIPMFSALSEGATPQDLNTMLNTVGGHQAAMQMLKETINEEAAEWDRVHPVHAGPIAPGQMREPRGSDIAGTTSTLQEQIGWMTNNPPIPVGEIYKRWIILGLNKIVRMYSPTSILDIRQGPKEPFRDYVDRFYKTLRAEQASQEVKNWMTETLLVQNANPDCKTILKALGPAATLEEMMTACQGVGGPGHKARVLAEAMSQVTNSATIMMQRGNFRNQRKIVKCFNCGKEGHIARNCRAPRKKGCWKCGKEGHQMKDCTERQANFLREDLAFLQGKAREFSSEQTRANSPTISSEQTRANSPTRRELQVWGRDNNSLSEAGADRQGTVSFNFPQITLWQRPLVTIKIGGQLKEALLDTGADDTVLEEMSLPGRWKPKMIGGIGGFIKVRQYDQILIEICGHKAIGTVLVGPTPVNIIGRNLLTQIGCTLNFPISPIETVPVKLKPGMDGPKVKQWPLTEEKIKALVEICTEMEKEGKISKIGPENPYNTPVFAIKKKDSTKWRKLVDFRELNKRTQDFWEVQLGIPHPAGLKKKKSVTVLDVGDAYFSVPLDEDFRKYTAFTIPSINNETPGIRYQYNVLPQGWKGSPAIFQSSMTKILEPFRKQNPDIVIYQYMDDLYVGSDLEIGQHRTKIEELRQHLLRWGLTTPDKKHQKEPPFLWMGYELHPDKWTVQPIVLPEKDSWTVNDIQKLVGKLNWASQIYPGIKVRQLCKLLRGTKALTEVIPLTEEAELELAENREILKEPVHGVYYDPSKDLIAEIQKQGQGQWTYQIYQEPFKNLKTGKYA

In [14]:
keys = {}
for i, sample in enumerate(data):
    name = sample["name"]
    if name not in keys:
        keys[name] = []
    keys[name].append(i)

for name in keys:
    if len(keys[name]) > 1:
        print(name, keys[name])

P03367_8473731aa5b9d8d2d7fbbe129137779d [90, 94]
P03367_e6d328e13e45dd811b1abb98042b5b5f [92, 96]
P03366_2c7d236bf0d1853656fb35f7e38ef07f [339, 357]
P03366_9a4be6c1d1e41cbe3169c9067e22b7d1 [341, 342]
P03366_220af7af3750f5439f5a575bf13af7ab [399, 400]
P05771_3fa7f12d4a7da52b8857584ff977cf7c [740, 748]
Q05513_8dc3cba7fe897cb6db3ad7234d4555ff [787, 788]
P00517_6361464f055c851451d5fd39ca95eb0b [795, 796]
P04585_0316b0bc375b7e8dec2ee7ed28d9f734 [2755, 2762]
P04585_c83ab3e3bffcac4d52db8f8597c8cd51 [2756, 2763, 2770]
P04585_7dad53b9c648966bdb60f0df5dc77006 [2758, 2765, 2772]
P04585_5218a0485e34d9eb053788f50dfb8467 [2759, 2766, 2773]
P04585_c65170d69d066bd8ce515abe155923f9 [2775, 2781, 2786]
P04585_bcd5f07cbdc102dee4cf46f1e4a5fe46 [2779, 2785, 2790]
P04585_a248c47b58ec0694680da817cb8a4555 [2795, 2801, 2805]
P04585_883b6863de878d42b68d1db9c1e4affc [2796, 2802, 2806]
P04585_eb3c937e1bb4ba52e93e0d1fd7a5a7a0 [2807, 2814]
P04585_5261968bfddee42a1cb2d6afa8537020 [2808, 2815, 2822]
P04585_503b145cb47

In [4]:
features = lmdb.get_split("feature")

In [5]:
features[0]

'p_P03367_10bf144af1f594e4669b66117d301b82'

In [6]:
x = lmdb['p_P03367_10bf144af1f594e4669b66117d301b82']

In [7]:
x.keys()

dict_keys(['feature_dict', 'atom_array'])

In [12]:
compressor = FeatureCompressor()

In [18]:
feature_dict = compressor.decompress(x['feature_dict'])

In [17]:
atom_array = x["atom_array"]
aa_tokenizer = AtomArrayTokenizer(atom_array)
token_array = aa_tokenizer.get_token_array()

In [22]:
lmdb_msa = LMDBDataset("/data/rerank/protenix/chembl_bdb/msa_feat.lmdb")

In [23]:
msa_keys = lmdb_msa.get_split("msa_filtered")

In [27]:
msa_feat = lmdb_msa[msa_keys[0]]
msa_feat = tokenize_msa(
    msa_feats=msa_feat,
    token_array=token_array,
    atom_array=atom_array,
)

In [29]:
msa_features = {
    k: v
    for (k, v) in msa_feat.items()
    if k
    in ["msa", "has_deletion", "deletion_value", "deletion_mean", "profile"]
}

In [32]:
res = dict_to_tensor(msa_features)

In [33]:
res

{'msa': tensor([[12, 15,  7,  ..., 20, 20, 20],
         [12, 15,  7,  ..., 20, 20, 20],
         [12, 15,  7,  ..., 20, 20, 20],
         ...,
         [31, 31, 31,  ..., 20, 20, 20],
         [31, 31, 31,  ..., 20, 20, 20],
         [31, 31, 31,  ..., 20, 20, 20]]),
 'profile': tensor([[0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
          9.7084e-01],
         [8.0451e-04, 0.0000e+00, 4.0225e-04,  ..., 0.0000e+00, 0.0000e+00,
          9.6199e-01],
         [8.0451e-04, 2.0113e-04, 4.0225e-04,  ..., 0.0000e+00, 0.0000e+00,
          9.4932e-01],
         ...,
         [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
          0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
          0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
          0.0000e+00]]),
 'deletion_mean': tensor([0.0000, 0.0000, 0.0004,  ..., 0.0000, 0.0000, 0.0000]),
 'has_deletion': tensor([[Fa